# Entrenar la U-Net de manganga-landmarks

Notebook listo para **Google Colab** (o Kaggle).

1. Activá GPU: `Entorno de ejecución → Cambiar tipo de entorno → T4 GPU`.
2. Ejecutá las celdas en orden.
3. Al final descargás `out/checkpoints/best.pt` y lo usás localmente con:

```bash
manganga detect -i ala.jpg -o out/ --model out/checkpoints/best.pt
```

In [ ]:
!nvidia-smi
import torch
print('CUDA disponible:', torch.cuda.is_available())
print('PyTorch:', torch.__version__)

In [ ]:
# El repo incluye el dataset (train/ masks/ test/ test_masks/) -> --depth 1 para que sea rápido
!git clone --depth 1 https://github.com/ensesovezz/manganga-landmarks.git
%cd manganga-landmarks

In [ ]:
%pip install -q typer rich tqdm opencv-python
%pip install -q -e . --no-deps
!manganga version

In [ ]:
# Chequeo rápido del dataset
!ls train | wc -l
!ls masks | wc -l
!ls test | wc -l
!ls test_masks | wc -l

In [ ]:
# Validar el ground truth: genera los manifiestos que excluyen las máscaras rotas
# (~7% de masks/ no tienen los 19 discos: el upstream las generó mal)
!manganga validate-data --mask-dir masks -o out
!manganga validate-data --mask-dir test_masks -o out

In [ ]:
# Entrenamiento completo (~40 épocas en T4: 1-3 h)
# Si querés guardar en Drive, montá Drive y cambiá -o por /content/drive/MyDrive/manganga/checkpoints
!manganga train --data-dir . --epochs 40 --batch-size 16 --size 400 --base 32 --depth 4 \
    --lr 1e-3 --pos-weight 50 --metric-every 5 --metric-samples 300 \
    --filter-dir out --device cuda --workers 2 -o out/checkpoints

In [ ]:
# Evaluación final sobre el test set completo (o una muestra)
from pathlib import Path
from manganga.data.dataset import list_pairs
from manganga.detect import load_checkpoint
from manganga.landmarks import load_reference
from manganga.train import evaluate_detection

model, config = load_checkpoint('out/checkpoints/best.pt', device='cuda')
reference = load_reference()
pairs = list_pairs('test', 'test_masks')[:500]

metrics = evaluate_detection(model, pairs, reference, size=config.get('size', 400),
                             device='cuda', max_samples=500)
metrics

In [ ]:
# Descargar el checkpoint y el log
!zip -q -r checkpoints.zip out/checkpoints
from google.colab import files
files.download('checkpoints.zip')

## Uso local del checkpoint

```bash
unzip checkpoints.zip -d out/       # queda out/checkpoints/best.pt
manganga batch -i fotos/ -o out/ --model out/checkpoints/best.pt
```

El `training_log.csv` que está dentro del zip tiene la evolución de la métrica
(% de alas con los 19 puntos detectados y ordenados). Referencia del paper: 0.9178.